Sandbox for experimenting with Auto3DSeg parameter passing.

Run this, let a fold start, Ctrl+C, then inspect the fold directory to see
what actually made it into hyper_parameters.yaml and training.log.

Usage:
    python scratch/auto3dseg_sandbox.py

Change WORK_DIR and the input_cfg / train_param / input_extras dicts below,
then rerun to try a new combination. Each run in a fresh directory starts clean.

In [10]:
import os
from pathlib import Path
# from monai.apps.auto3dseg import (
#     AutoRunner,
#     DataAnalyzer,
#     BundleGen,
#     import_bundle_algo_history,
#     export_bundle_algo_history,
# )
# from monai.auto3dseg import algo_to_pickle
# from monai.bundle.config_parser import ConfigParser
# from monai.utils.enums import AlgoKeys
from scripts import inspect_run
from core.configs import TrainingConfig
import yaml

## Inspecting Auto3dSeg

In [ ]:
fold_names = {
    1: "segresnet_0",
    2: "segresnet_1",
    3: "segresnet_2",
    4: "segresnet_3",
}
run_dir = Path("/home/shridhar.singh9-umw/prl_project/training/roi_train2/stage2_numcrops_dicece/run1")
run_dir = Path("/home/shridhar.singh9-umw/prl_project/training/roi_train2/stage4_sweep_dicece_wts/run1")
fold = 1
foldroot = run_dir / fold_names[fold]
log_path = foldroot / "model/training.log"
values = inspect_run.parse_training_log(log_path)

hyperparam_filepath = foldroot / "configs/hyper_parameters.yaml"
with open(hyperparam_filepath, 'r') as f:
    hyperparams = yaml.full_load(f)

In [14]:
print(hyperparams)
print(values)

{'_meta_': {}, 'bundle_root': '/home/shridhar.singh9-umw/prl_project/training/roi_train2/stage2_numcrops_dicece/run1/segresnet_0', 'ckpt_path': "$@bundle_root + '/model'", 'mlflow_tracking_uri': '/home/shridhar.singh9-umw/prl_project/training/roi_train2/stage2_numcrops_dicece/run1/mlruns', 'mlflow_experiment_name': 'run1', 'data_file_base_dir': '/home/shridhar.singh9-umw/prl_project/data', 'data_list_file_path': '/home/shridhar.singh9-umw/prl_project/training/roi_train2/stage2_numcrops_dicece/run1/datalist_xy20_z2.json', 'modality': 'mri', 'fold': 0, 'input_channels': 2, 'output_classes': 3, 'class_names': None, 'class_index': None, 'debug': False, 'ckpt_save': True, 'cache_rate': None, 'roi_size': [44, 44, 8], 'auto_scale_allowed': True, 'auto_scale_batch': True, 'auto_scale_roi': False, 'auto_scale_filters': False, 'quick': False, 'channels_last': True, 'validate_final_original_res': True, 'calc_val_loss': False, 'amp': True, 'log_output_file': None, 'cache_class_indices': None, 'ear

In [11]:
tr_params = TrainingConfig.from_dict(hyperparams)

In [ ]:
from monai.auto3dseg.utils import algo_from_pickle

pkl_filename = foldroot / "model/"

## Running Auto3dSeg

### Config

In [5]:
DATALIST  = "/home/srs-9/Projects/prl_project/training/roi_train2/datalist_flair.phase_xy20_z2.json"
DATAROOT  = "/media/smbshare/srs-9/prl_project/data"
WORK_DIR  = "/media/smbshare/srs-9/prl_project/training/roi_train2/test_runs"   # ← change this between experiments

# Params that go into the input dict → fill_template_config → hyper_parameters.yaml
# These survive regardless of type (scalars, lists, dicts all work here)
input_extras = {
    "roi_size": [44, 44, 8],
    "learning_rate": 0.0002,
    "num_epochs": 10,           # small for quick inspection
    "num_warmup_epochs": 1,
    "num_epochs_per_validation": 1,
    "auto_scale_batch": False,
    "num_images_per_batch": 1,
    # "batch_size": 2,            # → swapped to num_crops_per_image by segmenter
    "num_crops_per_image": 4
    # "crop_ratios": [1, 1, 4],
    # "loss": {
    #     "_target_": "DiceCELoss",
    #     "include_background": False,
    #     "weight": "$torch.tensor([1.0, 1.0, 5.0]).cuda()",
    #     "squared_pred": True,
    #     "smooth_nr": 0,
    #     "smooth_dr": 1e-05,
    #     "softmax": True,
    #     "sigmoid": False,
    #     "to_onehot_y": True,
    # },
}

# Params that go through set_training_params → CLI overrides (scalars only)
# For this project, leave empty — everything goes through input_extras above
train_param = {}

### Setup

In [6]:
os.makedirs(WORK_DIR, exist_ok=True)

input_cfg = {
    "modality": "MRI",
    "datalist": DATALIST,
    "dataroot": DATAROOT,
    **input_extras,
}

input_yaml = os.path.join(WORK_DIR, "input.yaml")
ConfigParser.export_config_file(input_cfg, input_yaml)
print(f"Wrote input.yaml to {input_yaml}")
print("input_cfg:", input_cfg)


Wrote input.yaml to /media/smbshare/srs-9/prl_project/training/roi_train2/test_runs/input.yaml
input_cfg: {'modality': 'MRI', 'datalist': '/home/srs-9/Projects/prl_project/training/roi_train2/datalist_flair.phase_xy20_z2.json', 'dataroot': '/media/smbshare/srs-9/prl_project/data', 'roi_size': [44, 44, 8], 'learning_rate': 0.0002, 'num_epochs': 10, 'num_warmup_epochs': 1, 'num_epochs_per_validation': 1, 'auto_scale_batch': False, 'num_images_per_batch': 1, 'num_crops_per_image': 4}


### Option A: AutoRunner

In [7]:
runner = AutoRunner(
    work_dir=WORK_DIR,
    algos=["segresnet"],
    input=input_cfg,
)
if train_param:
    runner.set_training_params(train_param)
runner.run()

2026-03-23 22:29:53,312 - INFO - AutoRunner using work directory /media/smbshare/srs-9/prl_project/training/roi_train2/test_runs
2026-03-23 22:29:53,354 - INFO - Datalist was copied to work_dir: /media/smbshare/srs-9/prl_project/training/roi_train2/test_runs/datalist_flair.phase_xy20_z2.json
2026-03-23 22:29:53,373 - INFO - Found num_fold 5 based on the input datalist /media/smbshare/srs-9/prl_project/training/roi_train2/test_runs/datalist_flair.phase_xy20_z2.json.
2026-03-23 22:29:53,373 - INFO - Setting num_fold 5 based on the input datalist /media/smbshare/srs-9/prl_project/training/roi_train2/test_runs/datalist_flair.phase_xy20_z2.json.
2026-03-23 22:29:53,397 - INFO - Using user defined command running prefix , will override other settings
2026-03-23 22:29:53,398 - INFO - Running data analysis...
2026-03-23 22:29:53,401 - INFO - Found 1 GPUs for data analyzing!


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
100%|██████████| 794/794 [00:10<00:00, 74.42it/s]


2026-03-23 22:30:04,492 - INFO - Data spacing is not completely uniform. MONAI transforms may provide unexpected result
2026-03-23 22:30:04,493 - INFO - Writing data stats to /media/smbshare/srs-9/prl_project/training/roi_train2/test_runs/datastats.yaml.
2026-03-23 22:30:04,513 - INFO - Writing by-case data stats to /media/smbshare/srs-9/prl_project/training/roi_train2/test_runs/datastats_by_case.yaml, this may take a while.
2026-03-23 22:30:07,132 - INFO - BundleGen from https://github.com/Project-MONAI/research-contributions/releases/download/algo_templates/21ed8e5.tar.gz


algo_templates.tar.gz: 104kB [00:00, 614kB/s]                              

2026-03-23 22:30:07,547 - INFO - Downloaded: /tmp/tmptpzz6qp4/algo_templates.tar.gz
2026-03-23 22:30:07,547 - INFO - Expected md5 is None, skip md5 check for file /tmp/tmptpzz6qp4/algo_templates.tar.gz.
2026-03-23 22:30:07,555 - INFO - Writing into directory: /media/smbshare/srs-9/prl_project/training/roi_train2/test_runs.


2026-03-23 22:30:09,939 - INFO - Generated:/media/smbshare/srs-9/prl_project/training/roi_train2/test_runs/segresnet_0
2026-03-23 22:30:10,710 - INFO - Generated:/media/smbshare/srs-9/prl_project/training/roi_train2/test_runs/segresnet_1
2026-03-23 22:30:11,476 - INFO - Generated:/media/smbshare/srs-9/prl_project/training/roi_train2/test_runs/segresnet_2
2026-03-23 22:30:12,268 - INFO - Generated:/media/smbshare/srs-9/prl_project/training/roi_train2/test_runs/segresnet_3
2026-03-23 22:30:13,065 - INFO - Generated:/media/smbshare/srs-9/prl_project/training/roi_train2/test_runs/segresnet_4
2026-03-23 22:30:13,328 - INFO - ['python', '/media/smbshare/srs-9/prl_project/training/roi_train2/test_runs/segresnet_0/scripts/train.py', 'run', "--config_file='/media/smbshare/srs-9/prl_project/training/roi_train2/test_runs/segresnet_0/configs/hyper_parameters.yaml'"]


<frozen importlib._bootstrap_external>:1324: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
monai.metrics.meandice DiceHelper.__init__:sigmoid: Argument `sigmoid` has been deprecated since version 1.5. It will be removed in version 1.7. Use `threshold` instead.
monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs

KeyboardInterrupt: 

### Option B: Step by step

In [ ]:
datastats_file = os.path.join(WORK_DIR, "datastats.yaml")
if not os.path.exists(datastats_file):
    analyser = DataAnalyzer(DATALIST, DATAROOT, output_path=datastats_file)
    analyser.get_all_case_stats()

bundle_gen = BundleGen(
    algo_path=WORK_DIR,
    data_stats_filename=datastats_file,
    data_src_cfg_name=input_yaml,
)
bundle_gen.generate(WORK_DIR, num_fold=5)
history = bundle_gen.get_history()
export_bundle_algo_history(history)

history = import_bundle_algo_history(WORK_DIR, only_trained=False)
for algo_dict in history:
    algo = algo_dict[AlgoKeys.ALGO]
    algo.train(train_param or {})
    try:
        acc = algo.get_score()
        algo_to_pickle(algo, template_path=algo.template_path, best_metric=acc)
    except Exception:
        pass  # fold killed before completion, score unavailable